# REPA — Comparison Notebook (Colab)

Runs all 4 combinations and prints a FID comparison table.

**Before running:**
1. Upload your `meddinov3_minimal/` folder to Google Drive root
2. Upload your `kaggle.json` to Google Drive root (for CheXpert download)
3. Runtime → Change runtime type → **T4 GPU**

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.system('tar -xzf /content/meddinov3_minimal.tar.gz -C /content/')
print('MedDINOv3 extracted:', os.path.isdir('/content/meddinov3_minimal'))

## 2. Install dependencies

In [ ]:
%%bash
pip install -q accelerate diffusers timm einops pandas torch-fidelity kaggle

## 3. Download CheXpert via Kaggle API

In [ ]:
import os, shutil

# copy kaggle.json from Drive
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('/content/drive/MyDrive/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# download CheXpert-v1.0-small (~11 GB, takes ~10 min)
if not os.path.isdir('/content/chexpert'):
    os.makedirs('/content/chexpert', exist_ok=True)
    os.system('kaggle datasets download -d ashery/chexpert -p /content/chexpert --unzip')

print(f"train.csv exists: {os.path.isfile('/content/chexpert/train.csv')}")

## 4. Clone REPA fork

In [ ]:
%%bash
if [ ! -d "/content/REPA" ]; then
    git clone https://github.com/nikiboura/REPA.git /content/REPA
fi
echo "REPA ready. Commit: $(git -C /content/REPA rev-parse --short HEAD)"

## 5. Set paths

In [ ]:
import os

MEDDINOV3_PATH = '/content/meddinov3_minimal'
MEDDINOV3_CKPT = MEDDINOV3_PATH + '/checkpoints/model.pth'
CHEXPERT_ROOT  = '/content/chexpert'

os.environ['MEDDINOV3_PATH'] = MEDDINOV3_PATH
os.environ['MEDDINOV3_CKPT'] = MEDDINOV3_CKPT

print(f'MedDINOv3 exists  : {os.path.isdir(MEDDINOV3_PATH)}')
print(f'Checkpoint exists : {os.path.isfile(MEDDINOV3_CKPT)}')
print(f'CheXpert exists   : {os.path.isdir(CHEXPERT_ROOT)}')
print(f'train.csv exists  : {os.path.isfile(os.path.join(CHEXPERT_ROOT, "train.csv"))}')

## 6. Prepare CIFAR-10

In [ ]:
import subprocess, sys

result = subprocess.run([
    sys.executable, '/content/REPA/prepare_cifar10.py',
    '--out-dir',     '/content/data/cifar10_256',
    '--resolution',  '256',
    '--max-samples', '5000',
], text=True)
print('Exit code:', result.returncode)

## 7. Prepare CheXpert

In [ ]:
import subprocess, sys

result = subprocess.run([
    sys.executable, '/content/REPA/prepare_chexpert.py',
    '--out-dir',       '/content/data/chexpert_256',
    '--chexpert-root', CHEXPERT_ROOT,
    '--resolution',    '256',
    '--max-samples',   '5000',
], text=True)
print('Exit code:', result.returncode)

## 8. Run all combinations
Trains, generates and computes FID for each combination sequentially (~8 hours total).

In [ ]:
import subprocess, os, re, glob
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm as tqdm_nb

COMBINATIONS = [
    ('meddinov3', 'cifar10'),
    ('meddinov3', 'chexpert'),
    ('dinov2',    'cifar10'),
    ('dinov2',    'chexpert'),
]

results = {}

for ENCODER, DATASET in COMBINATIONS:
    print(f'\n{"="*50}')
    print(f'  {ENCODER.upper()} x {DATASET.upper()}')
    print(f'{"="*50}')

    EXP_NAME     = f'repa-sits8-{ENCODER}-{DATASET}'
    NUM_CLASSES  = '10' if DATASET == 'cifar10' else '2'
    DATA_DIR     = f'/content/data/{DATASET}_256'
    ENC_TYPE     = 'dinov2-vit-b' if ENCODER == 'dinov2' else 'meddinov3-vit-b'
    FID_REAL_DIR = DATA_DIR + ('/images/val' if DATASET == 'cifar10' else '/images/train')

    env = os.environ.copy()
    if ENCODER == 'meddinov3':
        env['MEDDINOV3_PATH'] = MEDDINOV3_PATH
        env['MEDDINOV3_CKPT'] = MEDDINOV3_CKPT

    # ── Train ──────────────────────────────────────────────────
    print('Training...')
    train_cmd = [
        'accelerate', 'launch',
        '--mixed_precision', 'fp16',
        '--num_processes', '1',
        '/content/REPA/train.py',
        '--exp-name',            EXP_NAME,
        '--output-dir',          '/content/results',
        '--report-to',           'none',
        '--model',               'SiT-S/8',
        '--num-classes',         NUM_CLASSES,
        '--encoder-depth',       '8',
        '--enc-type',            ENC_TYPE,
        '--data-dir',            DATA_DIR,
        '--resolution',          '256',
        '--batch-size',          '32',
        '--num-workers',         '2',
        '--epochs',              '200',
        '--learning-rate',       '1e-4',
        '--mixed-precision',     'fp16',
        '--proj-coeff',          '0.5',
        '--cfg-prob',            '0.1',
        '--path-type',           'linear',
        '--max-train-steps',     '15000',
        '--checkpointing-steps', '15000',
        '--sampling-steps',      '99999999',
    ]
    process = subprocess.Popen(train_cmd, cwd='/content/REPA', env=env,
                               stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
    pbar = tqdm_nb(total=15000, desc=f'{ENCODER} x {DATASET}')
    last_step = 0
    for line in process.stdout:
        m = re.search(r'Steps:\s*(\d+)/\d+.*?loss=([\d.]+).*?proj_loss=(-?[\d.]+)', line)
        if m:
            step = int(m.group(1))
            if step > last_step:
                pbar.update(step - last_step)
                pbar.set_postfix({'loss': m.group(2), 'proj': m.group(3)})
                last_step = step
    pbar.close()
    process.wait()
    print('Training done.')

    # ── Generate ───────────────────────────────────────────────
    print('Generating 2000 samples...')
    ckpts = sorted(glob.glob(f'/content/results/{EXP_NAME}/checkpoints/*.pt'))
    gen_cmd = [
        'torchrun', '--nproc_per_node=1',
        '/content/REPA/generate.py',
        '--model',                'SiT-S/8',
        '--ckpt',                 ckpts[-1],
        '--encoder-depth',        '8',
        '--num-classes',          NUM_CLASSES,
        '--projector-embed-dims', '768',
        '--per-proc-batch-size',  '16',
        '--num-fid-samples',      '2000',
        '--path-type',            'linear',
        '--mode',                 'ode',
        '--num-steps',            '50',
        '--cfg-scale',            '1.0',
        '--resolution',           '256',
        '--vae',                  'mse',
    ]
    subprocess.run(gen_cmd, cwd='/content/REPA', env=env,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print('Generation done.')

    # ── FID ────────────────────────────────────────────────────
    print('Computing FID...')
    gen_npz  = sorted(glob.glob('/content/REPA/samples/**/*.npz', recursive=True))[-1]
    gen_dir  = f'/content/gen_{ENCODER}_{DATASET}'
    os.makedirs(gen_dir, exist_ok=True)
    npz_data = np.load(gen_npz)
    imgs     = npz_data[npz_data.files[0]]
    for i, img in enumerate(imgs):
        Image.fromarray(img.astype('uint8')).save(f'{gen_dir}/{i:05d}.png')

    fid_out = subprocess.run([
        'fidelity', '--gpu', '0', '--fid',
        '--input1', gen_dir,
        '--input2', FID_REAL_DIR,
    ], capture_output=True, text=True)

    m = re.search(r'frechet_inception_distance:\s*([\d.]+)', fid_out.stdout)
    fid_score = float(m.group(1)) if m else float('nan')
    results[(ENCODER, DATASET)] = fid_score
    print(f'FID: {fid_score}')

print('\nAll combinations done.')

## 9. Results

In [ ]:
print()
print('='*45)
print(f"{'RESULTS':^45}")
print('='*45)
print(f"{'Encoder':<15} {'Dataset':<12} {'FID':>10}")
print('-'*45)
for (enc, ds), fid in results.items():
    print(f'{enc:<15} {ds:<12} {fid:>10.4f}')
print('='*45)